# Event-Phase TFT Training with Optuna — NG EIA Storage Releases

ClassificationEventTFT on Natural Gas around weekly EIA storage report releases.

Each sample = one 5-min bar after the post-event window ends, carrying
the event day's phase orderflow (pre/event/post) as shared context.
Rolling 30-min forward returns classified into ordinal buckets.
Position derived from class probabilities, optimized via CE + ContinuousTradingLoss.

**Expects:**
- `/content/drive/MyDrive/features/NG/intraday.csv`
- `/content/drive/MyDrive/features/NG/ng_release_orderflow.parquet`
- `/content/drive/MyDrive/features/NG/ng_release_stats.csv`

In [ ]:
from __future__ import annotations

import gc
import math
from datetime import date
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

In [ ]:
from CTAFlow.models.prep.event_release_dataset import (
    EventReleaseQuantilePrep,
    build_session_samples,
    EventSessionDataset,
    event_session_collate_fn,
)
from CTAFlow.models.deep_learning.multi_branch.tft.event_phase_tft import (
    EventPhaseTFTConfig,
    ClassificationEventTFTConfig,
    ClassificationEventTFT,
)
from CTAFlow.models.deep_learning.training.loss.clf import (
    ContinuousTradingLoss,
    SharpeScheduler,
)

## Config

In [ ]:
# Try Colab path first, fall back to local
COLAB_ROOT = Path("/content/drive/MyDrive/features/NG")
LOCAL_ROOT = Path("F:/Upload/s3/model_data/NG")
DATA_ROOT = COLAB_ROOT if COLAB_ROOT.exists() else LOCAL_ROOT

OHLCV_PATH = DATA_ROOT / "intraday.csv"
ORDERFLOW_PATH = DATA_ROOT / "ng_release_orderflow.parquet"
STATS_PATH = DATA_ROOT / "ng_release_stats.csv"

# Training
NUM_EPOCHS = 20
WARMUP_EPOCHS = 5
N_TRIALS = 30
VAL_CUTOFF = date(2023, 1, 1)  # split on actual sample date
USE_AMP = True
STRIDE = 6  # non-overlapping 30-min windows (6 x 5min)
HORIZON_BARS = 6
SESSION_CLOSE = "17:00"
INCLUDE_NEXT_DAY_RTH = True
NEXT_DAY_RTH_START = "08:30"
NEXT_DAY_RTH_END = "13:00"
FIXED_LENGTH = 128  # orderflow buckets per phase

# Selection score
SELECTION_SHARPE_WEIGHT = 0.20
SELECTION_SORTINO_WEIGHT = 0.35
SELECTION_PF_WEIGHT = 0.25
SELECTION_RETURN_WEIGHT = 0.20
SELECTION_TOTAL_RETURN_SCALE = 100.0

# Orderflow feature columns (exclude metadata: bucket, date, event_code, side)
ORDERFLOW_FEATURE_COLS = [
    "buy", "sell", "vol", "close", "imbalance",
    "imb_frac", "vpin", "bucket_return", "log_duration",
    "signed_imbalance", "buy_dom", "sell_dom",
    "max_buy_run", "max_sell_run", "vol_ratio", "bucket_volume",
]
F_PHASE = len(ORDERFLOW_FEATURE_COLS)  # 16
NUM_PHASES = 3  # pre, event, post

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print("Selection score weights:")
print(
    f"  sharpe={SELECTION_SHARPE_WEIGHT}, sortino={SELECTION_SORTINO_WEIGHT}, "
    f"pf={SELECTION_PF_WEIGHT}, total_return={SELECTION_RETURN_WEIGHT}"
)


## Data Loading

In [ ]:
def load_orderflow_phases(
    orderflow_path: Path,
    feature_cols: List[str],
    fixed_length: int = 128,
) -> Dict[date, np.ndarray]:
    """Load orderflow parquet -> {date: (3, 128, F)} phase arrays.

    Groups by (date, side) where side in {pre, event, post}.
    """
    df = pd.read_parquet(orderflow_path)
    df["date"] = pd.to_datetime(df["date"]).dt.date

    phase_map = {"pre": 0, "event": 1, "post": 2}
    result: Dict[date, np.ndarray] = {}

    for dt, day_group in df.groupby("date"):
        phases = np.zeros((3, fixed_length, len(feature_cols)), dtype=np.float32)
        for side, side_group in day_group.groupby("side"):
            if side not in phase_map:
                continue
            p_idx = phase_map[side]
            arr = side_group[feature_cols].values.astype(np.float32)
            n = min(arr.shape[0], fixed_length)
            phases[p_idx, :n, :] = arr[:n]
        result[dt] = phases

    return result


def load_post_end_times(
    stats_path: Path,
    post_minutes: int = 60,
) -> Dict[date, pd.Timestamp]:
    """Derive post-window end times from event stats CSV.

    EIA NG storage releases at 10:30 ET (09:30 CT).
    post_minutes=60 -> post ends at 10:30 CT.
    """
    stats = pd.read_csv(stats_path, index_col=[0, 1], parse_dates=False)
    post_end: Dict[date, pd.Timestamp] = {}

    for (dt_str, _code), _row in stats.iterrows():
        dt = pd.Timestamp(dt_str).date()
        if dt in post_end:
            continue
        release_ct = pd.Timestamp(f"{dt} 09:30:00")
        post_end[dt] = release_ct + pd.Timedelta(minutes=post_minutes)

    return post_end

In [ ]:
def build_all_samples():
    """Build all per-bar samples from orderflow + intraday data."""
    print("Loading orderflow phases...")
    phase_orderflow = load_orderflow_phases(
        ORDERFLOW_PATH, ORDERFLOW_FEATURE_COLS, FIXED_LENGTH
    )
    print(f"  {len(phase_orderflow)} event days with orderflow")

    print("Loading post-end times...")
    post_end_times = load_post_end_times(STATS_PATH)
    print(f"  {len(post_end_times)} post-end times")

    print("Loading OHLCV + building prep...")
    prep = EventReleaseQuantilePrep(
        horizon_bars=HORIZON_BARS,
        quantiles=(0.25, 0.5, 0.75),
        min_quantile_history=40,
        enable_single_point_classification=True,
        single_point_target_step=HORIZON_BARS,
        single_point_class_quantiles=(0.25, 0.5, 0.75),
    )
    prep.load_data(ohlcv_csv_path=str(OHLCV_PATH))
    print(f"  {len(prep.frame)} bars in OHLCV")

    event_dates = sorted(phase_orderflow.keys())
    print(f"Building per-bar samples (stride={STRIDE})...")
    samples = build_session_samples(
        prep,
        event_dates=event_dates,
        post_end_times=post_end_times,
        phase_orderflow=phase_orderflow,
        session_close_time=SESSION_CLOSE,
        stride=STRIDE,
        horizon_bars=HORIZON_BARS,
        include_next_day_rth=INCLUDE_NEXT_DAY_RTH,
        next_day_rth_start=NEXT_DAY_RTH_START,
        next_day_rth_end=NEXT_DAY_RTH_END,
    )
    print(f"  {len(samples)} total samples")

    if samples:
        f_bar = len(samples[0]["bar_features"])
        print(f"  bar_feature_dim = {f_bar}")
    else:
        raise ValueError("No samples built -- check data paths and event dates")

    return samples, f_bar


In [ ]:
ALL_SAMPLES, F_BAR = build_all_samples()

TRAIN_SAMPLES = [s for s in ALL_SAMPLES if s["date"] < VAL_CUTOFF]
VAL_SAMPLES = [s for s in ALL_SAMPLES if s["date"] >= VAL_CUTOFF]
print(f"Train: {len(TRAIN_SAMPLES)}  |  Val: {len(VAL_SAMPLES)}")

## Train / Eval Loops

In [ ]:
def train_epoch(
    model: ClassificationEventTFT,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    *,
    max_norm: float = 1.0,
    scaler: Optional[torch.amp.GradScaler] = None,
) -> Tuple[float, Dict[str, float]]:
    model.train()
    total_loss = 0.0
    metrics_sum: Dict[str, float] = {}
    n_batches = 0

    for batch in loader:
        phase_features = batch["phase_features"].to(device)
        bar_features = batch["bar_features"].to(device)
        target_return = batch["target_return"].to(device)
        target_class = batch["target_class"].to(device)
        is_last_bar = batch["is_last_bar"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device.type, enabled=scaler is not None):
            outputs = model(
                phase_features=phase_features,
                bar_features=bar_features,
            )
            losses = model.compute_loss(
                outputs,
                target_returns=target_return,
                target_classes=target_class,
                is_last_bar=is_last_bar,
            )
            loss = losses["total_loss"]

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm)
            optimizer.step()

        total_loss += loss.item()
        for k, v in losses.items():
            if isinstance(v, torch.Tensor) and v.dim() == 0:
                metrics_sum[k] = metrics_sum.get(k, 0.0) + v.item()
        n_batches += 1

    avg_loss = total_loss / max(n_batches, 1)
    avg_metrics = {k: v / max(n_batches, 1) for k, v in metrics_sum.items()}
    return avg_loss, avg_metrics


@torch.no_grad()
def evaluate(
    model: ClassificationEventTFT,
    loader: DataLoader,
) -> Dict[str, float]:
    model.eval()
    all_positions = []
    all_returns = []
    all_preds = []
    all_targets = []
    metrics_sum: Dict[str, float] = {}
    n_batches = 0

    for batch in loader:
        phase_features = batch["phase_features"].to(device)
        bar_features = batch["bar_features"].to(device)
        target_return = batch["target_return"].to(device)
        target_class = batch["target_class"].to(device)
        is_last_bar = batch["is_last_bar"].to(device)

        outputs = model(
            phase_features=phase_features,
            bar_features=bar_features,
        )
        losses = model.compute_loss(
            outputs,
            target_returns=target_return,
            target_classes=target_class,
            is_last_bar=is_last_bar,
        )

        for k, v in losses.items():
            if isinstance(v, torch.Tensor) and v.dim() == 0:
                metrics_sum[k] = metrics_sum.get(k, 0.0) + v.item()

        all_positions.append(outputs["position"].detach().cpu())
        all_returns.append(target_return.detach().cpu())
        all_preds.append(outputs["logits"].argmax(dim=-1).detach().cpu())
        all_targets.append(target_class.detach().cpu())
        n_batches += 1

    avg_metrics = {k: v / max(n_batches, 1) for k, v in metrics_sum.items()}
    avg_metrics["loss"] = avg_metrics.get("total_loss", 0.0)

    if not all_positions:
        return {
            **avg_metrics,
            "sharpe": 0.0,
            "sortino": 0.0,
            "profit_factor": 0.0,
            "mean_strategy_ret": 0.0,
            "win_rate": 0.0,
            "dir_accuracy": 0.0,
            "avg_exposure": 0.0,
            "max_drawdown": 0.0,
            "downside_vol": 0.0,
            "accuracy": 0.0,
            "exposure": 0.0,
            "dir_acc": 0.0,
        }

    positions = torch.cat(all_positions)
    returns = torch.cat(all_returns)
    preds = torch.cat(all_preds)
    targets = torch.cat(all_targets)
    pnl = positions * returns

    mean_ret = pnl.mean().item()
    pnl_std = pnl.std(unbiased=False).item()
    downside = pnl[pnl < 0]
    downside_vol = downside.std(unbiased=False).item() if downside.numel() > 0 else 0.0
    gross_profit = pnl[pnl > 0].sum().item()
    gross_loss = (-pnl[pnl < 0]).sum().item()
    profit_factor = gross_profit / (gross_loss + 1e-8)
    sharpe = mean_ret / (pnl_std + 1e-8)
    sortino = mean_ret / (downside_vol + 1e-8)
    win_rate = (pnl > 0).float().mean().item()
    avg_exposure = positions.abs().mean().item()

    correct_dir = ((positions > 0) & (returns > 0)) | ((positions < 0) & (returns < 0))
    active = positions.abs() > 0.05
    dir_accuracy = correct_dir[active].float().mean().item() if active.any().item() else 0.0

    cumulative = torch.cumsum(pnl, dim=0)
    running_max = torch.cummax(cumulative, dim=0).values
    max_drawdown = (running_max - cumulative).max().item() if cumulative.numel() > 0 else 0.0

    avg_metrics.update(
        {
            "sharpe": sharpe,
            "sortino": sortino,
            "profit_factor": profit_factor,
            "mean_strategy_ret": mean_ret,
            "win_rate": win_rate,
            "dir_accuracy": dir_accuracy,
            "avg_exposure": avg_exposure,
            "max_drawdown": max_drawdown,
            "downside_vol": downside_vol,
            "accuracy": (preds == targets).float().mean().item(),
            "exposure": avg_exposure,
            "dir_acc": dir_accuracy,
        }
    )
    return avg_metrics


def hybrid_selection_score(metrics: Dict[str, float], n_samples: int) -> float:
    sharpe = float(metrics.get("sharpe", 0.0))
    sortino = float(metrics.get("sortino", 0.0))
    profit_factor = float(metrics.get("profit_factor", 1e-8))
    mean_strategy_ret = float(metrics.get("mean_strategy_ret", 0.0))

    total_return = mean_strategy_ret * float(n_samples)
    total_return_score = math.copysign(
        math.log1p(abs(total_return) * SELECTION_TOTAL_RETURN_SCALE),
        total_return,
    )
    log_pf = math.log(max(profit_factor, 1e-8))

    return (
        SELECTION_SHARPE_WEIGHT * sharpe
        + SELECTION_SORTINO_WEIGHT * sortino
        + SELECTION_PF_WEIGHT * log_pf
        + SELECTION_RETURN_WEIGHT * total_return_score
    )


## Optuna Objective

In [ ]:
def objective(trial: optuna.Trial) -> float:
    # -- Architecture --
    d_model = trial.suggest_categorical("d_model", [32, 64, 128])
    d_hidden = d_model
    d_phase_emb = trial.suggest_categorical("d_phase_emb", [16, 32])
    d_static_emb = trial.suggest_categorical("d_static_emb", [16, 32])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    num_classes = trial.suggest_categorical("num_classes", [4, 5])

    # -- Training --
    batch_size = trial.suggest_categorical("batch_size", [32, 48, 64])
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-3, 7e-3, log=True)
    max_norm = trial.suggest_float("max_norm", 0.5, 1.0)

    # -- Trading loss --
    tc_cost = trial.suggest_float("tc_cost", 5e-5, 5e-4, log=True)
    init_direction_weight = trial.suggest_float("init_direction_weight", 0.5, 1.5)
    final_direction_weight = trial.suggest_float("final_direction_weight", 0.05, 0.3)
    init_reg_weight = trial.suggest_float("init_reg_weight", 0.1, 0.5)
    target_exposure = trial.suggest_float("target_exposure", 0.2, 0.5)
    downside_vol_weight = trial.suggest_float("downside_vol_weight", 0.02, 0.75, log=True)
    holding_weight = trial.suggest_float("holding_weight", 0.0, 0.5)
    exposure_asymmetry = trial.suggest_float("exposure_asymmetry", 1.0, 6.0)
    ce_weight = trial.suggest_float("ce_weight", 0.3, 2.0, log=True)
    trading_weight = trial.suggest_float("trading_weight", 0.3, 2.0, log=True)
    use_sortino = True

    print(
        f"Trial {trial.number:03d} | d_model={d_model} | heads={n_heads} | "
        f"batch={batch_size} | lr={learning_rate:.2e} | tc={tc_cost:.2e}"
    )

    trunk_cfg = EventPhaseTFTConfig(
        input_dim=F_PHASE,
        phase_seq_len=FIXED_LENGTH,
        horizon_steps=HORIZON_BARS,
        d_model=d_model,
        d_hidden=d_hidden,
        d_phase_emb=d_phase_emb,
        d_static_emb=d_static_emb,
        n_heads=n_heads,
        dropout=dropout,
    )
    config = ClassificationEventTFTConfig(
        trunk=trunk_cfg,
        num_classes=num_classes,
        bar_feature_dim=F_BAR,
        tc_cost=tc_cost,
        direction_weight=init_direction_weight,
        reg_weight=init_reg_weight,
        target_exposure=target_exposure,
        ce_weight=ce_weight,
        trading_weight=trading_weight,
        use_sortino=use_sortino,
    )
    model = ClassificationEventTFT(config).to(device)
    model.trading_loss_fn.downside_vol_weight = downside_vol_weight
    model.trading_loss_fn.holding_weight = holding_weight
    model.trading_loss_fn.exposure_asymmetry = exposure_asymmetry
    model.trading_loss_fn.tc_in_sharpe = True

    try:
        train_ds = EventSessionDataset(TRAIN_SAMPLES)
        val_ds = EventSessionDataset(VAL_SAMPLES)
        train_loader = DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=event_session_collate_fn,
            num_workers=0,
            drop_last=True,
        )
        val_loader = DataLoader(
            val_ds,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=event_session_collate_fn,
            num_workers=0,
        )
    except Exception as exc:
        print(f"Dataloader failed: {exc}")
        return -1e9

    optimizer = optim.AdamW(
        model.parameters(), lr=learning_rate, weight_decay=weight_decay
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    scaler = torch.amp.GradScaler() if (USE_AMP and device.type == "cuda") else None

    sharpe_sched = SharpeScheduler(
        warmup_epochs=WARMUP_EPOCHS,
        total_epochs=NUM_EPOCHS,
        initial_direction_weight=init_direction_weight,
        final_direction_weight=final_direction_weight,
        initial_reg_weight=init_reg_weight,
        final_reg_weight=0.05,
        initial_target_exposure=0.2,
        final_target_exposure=target_exposure,
        initial_holding_weight=0.0,
        final_holding_weight=holding_weight,
    )

    best_score = -1e9
    patience_counter = 0
    prev_val_loss = None
    n_val_samples = len(VAL_SAMPLES)

    for epoch in range(NUM_EPOCHS):
        sharpe_sched.step(epoch, model.trading_loss_fn)
        is_warmup = epoch < WARMUP_EPOCHS

        train_loss, train_metrics = train_epoch(
            model,
            train_loader,
            optimizer,
            max_norm=max_norm,
            scaler=scaler,
        )
        val_metrics = evaluate(model, val_loader)
        scheduler.step()

        val_loss = val_metrics["loss"]
        val_score = hybrid_selection_score(val_metrics, n_val_samples)

        if math.isnan(val_loss) or math.isinf(val_loss) or val_loss > 100.0:
            return best_score if best_score > -1e9 else -1e9
        if prev_val_loss is not None and epoch >= 3 and val_loss > abs(prev_val_loss) * 5.0:
            return best_score if best_score > -1e9 else -1e9
        prev_val_loss = val_loss

        if (epoch + 1) % 5 == 0:
            print(
                f"  E{epoch + 1:02d} | Score: {val_score:.4f} | Sharpe: {val_metrics['sharpe']:.4f} | "
                f"Sortino: {val_metrics['sortino']:.4f} | PF: {val_metrics['profit_factor']:.3f} | "
                f"MeanRet: {val_metrics['mean_strategy_ret']:.6f}{' [warmup]' if is_warmup else ''}"
            )

        if is_warmup:
            continue

        if val_score > best_score:
            best_score = val_score
            patience_counter = 0
            trial.set_user_attr("final_selection_score", val_score)
            trial.set_user_attr("final_sharpe", val_metrics["sharpe"])
            trial.set_user_attr("final_sortino", val_metrics["sortino"])
            trial.set_user_attr("final_mean_strategy_ret", val_metrics["mean_strategy_ret"])
            trial.set_user_attr("final_win_rate", val_metrics["win_rate"])
            trial.set_user_attr("final_dir_acc", val_metrics["dir_accuracy"])
            trial.set_user_attr("final_pf", val_metrics["profit_factor"])
            trial.set_user_attr("final_exposure", val_metrics["avg_exposure"])
            trial.set_user_attr("final_loss", val_loss)
            trial.set_user_attr("final_downside_vol", val_metrics.get("downside_vol", 0.0))
            trial.set_user_attr("final_accuracy", val_metrics.get("accuracy", 0.0))
        else:
            patience_counter += 1

        trial.report(val_score, epoch)
        if trial.should_prune():
            print(f"  Trial {trial.number:03d} pruned at epoch {epoch + 1:02d} (score={val_score:.4f})")
            raise optuna.TrialPruned()
        if patience_counter >= 8:
            break

    del model, optimizer, scheduler, scaler, train_loader, val_loader
    if device.type == "cuda":
        torch.cuda.empty_cache()
    gc.collect()
    return best_score


## Run Study

In [ ]:
study = optuna.create_study(
    study_name="event_tft_NG_EIA_classification_next_day_rth",
    direction="maximize",
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=WARMUP_EPOCHS),
)

print(f"Starting optimization: {N_TRIALS} trials")
print(f"Study: {study.study_name}")
print("Objective: composite selection score (Sharpe + Sortino + ProfitFactor + TotalReturn)")
print(
    f"Sample window: post-event through {SESSION_CLOSE} plus next-day US RTH "
    f"{NEXT_DAY_RTH_START}-{NEXT_DAY_RTH_END}"
)
print("-" * 60)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
    gc_after_trial=True,
)


## Results

In [ ]:
best_trial = study.best_trial
print(f"Best trial #{best_trial.number}:")
print(f"  Selection score: {best_trial.value:.6f}")
if best_trial.user_attrs:
    attr_keys = [
        "final_selection_score",
        "final_sharpe",
        "final_sortino",
        "final_mean_strategy_ret",
        "final_win_rate",
        "final_dir_acc",
        "final_pf",
        "final_exposure",
        "final_downside_vol",
        "final_accuracy",
        "final_loss",
    ]
    for key in attr_keys:
        print(f"  {key}: {best_trial.user_attrs.get(key, 'N/A')}")

print("  Params:")
for key, value in sorted(best_trial.params.items()):
    print(f"    {key}: {value}")


In [ ]:
import joblib

out_path = DATA_ROOT / "optuna_study.pkl"
joblib.dump(study, str(out_path))
print(f"Study saved -> {out_path}")

In [ ]:
# Optuna visualization (optional)
try:
    from optuna.visualization import (
        plot_optimization_history,
        plot_param_importances,
        plot_parallel_coordinate,
    )
    display(plot_optimization_history(study))
    display(plot_param_importances(study))
    display(plot_parallel_coordinate(study))
except ImportError:
    print("Install plotly for Optuna visualizations: pip install plotly")